In [1]:
from datetime import date, timedelta

import polars as pl

from data_engineering.adsb.read_adsb import read_icaos_from_adsb
from data_engineering.flights.flight_type import get_flights


start_date = date(2026, 5, 14)
end_date = date(2026, 5, 27)
target_dates = [
    start_date + timedelta(days=i)
    for i in range((end_date - start_date).days + 1)
]

df_flights = pl.concat([get_flights(target_date) for target_date in target_dates])
df_ladd_flights = df_flights.filter(pl.col("ladd").fill_null(False))

ladd_icaos = df_ladd_flights.get_column("icao").drop_nulls().unique().sort().to_list()
df_adsb = pl.concat([
    read_icaos_from_adsb(target_date, ladd_icaos)
    for target_date in target_dates
])

{
    "start_date": start_date,
    "end_date": end_date,
    "days": len(target_dates),
    "ladd_flights": len(df_ladd_flights),
    "ladd_icaos": len(ladd_icaos),
    "adsb_rows": len(df_adsb),
}


{'start_date': datetime.date(2026, 5, 14),
 'end_date': datetime.date(2026, 5, 27),
 'days': 14,
 'ladd_flights': 110365,
 'ladd_icaos': 12024,
 'adsb_rows': 58113550}

In [2]:
from flights.flights_match_adsb import flights_match_adsb


match_results = []
for icao in ladd_icaos:
    icao_flights = df_ladd_flights.filter(pl.col("icao") == icao).sort("takeoff_time")
    icao_adsb = df_adsb.filter(pl.col("icao") == icao).sort("time")

    matches = flights_match_adsb(icao_flights, icao_adsb)

    match_results.append({
        "icao": icao,
        "matches": matches,
        "flight_rows": len(icao_flights),
        "adsb_rows": len(icao_adsb),
    })

df_match_results = pl.DataFrame(match_results)
match_counts = df_match_results.group_by("matches").len().sort("matches", descending=True)
true_matches = df_match_results.filter(pl.col("matches")).height
total_icaos = df_match_results.height

{
    "true_matches": true_matches,
    "false_matches": df_match_results.filter(~pl.col("matches")).height,
    "total_icaos": total_icaos,
    "true_match_percentage": true_matches / total_icaos * 100,
}

{'true_matches': 3488,
 'false_matches': 8536,
 'total_icaos': 12024,
 'true_match_percentage': 29.008649367930804}

In [3]:
matched_icaos = df_match_results.filter(pl.col("matches")).get_column("icao").to_list()
df_ladd_flights_matched_adsb = df_ladd_flights.filter(pl.col("icao").is_in(matched_icaos))

{
    "matched_ladd_flights": len(df_ladd_flights_matched_adsb),
    "matched_ladd_icaos": len(matched_icaos),
}


{'matched_ladd_flights': 13927, 'matched_ladd_icaos': 3488}